# Week 5 — Apache Spark Fundamentals: Data Cleaning, Transformation & Aggregation

**Objective:** Understand Spark fundamentals and perform data cleaning, transformation, and aggregation using DataFrames.

**What this notebook covers:**
- Why Spark is preferred over traditional MapReduce
- Spark DataFrame concepts and immutability
- Data cleaning (duplicates, nulls, inconsistent data)
- Filtering, transformation, schema changes
- Aggregation and grouping
- Wide transformations and the shuffle
- A complete end-to-end data processing pipeline
- Answers to all 15 assignment questions (Q1–Q15), each with runnable code and real output

**Dataset:** A synthetic e-commerce/transactions dataset (`data/dataset.csv`) was generated for this assignment, containing the columns referenced across the assignment brief: `user_id`, `transaction_date`, `region`, `product_category`, `sale_amount`, `age`, `subscription`, `city`, `price`, `store_id`, `email`, `username`, `status`, `raw_timestamp`. It includes intentional duplicates, null values, and an empty-string username to realistically exercise every cleaning step below.


## Step 1: Spark vs MapReduce — Background

**MapReduce limitations:**
- Reads/writes intermediate results to disk after every map and reduce stage → high I/O overhead
- Poor performance for iterative algorithms (e.g. machine learning) since each iteration re-reads from disk
- Verbose, low-level programming model
- No interactive/ad-hoc querying — every job is a full batch job

**Spark advantages:**
- **In-memory computing**: intermediate data stays in RAM (via the Resilient Distributed Dataset / DataFrame abstraction) across operations, avoiding repeated disk I/O
- **DAG execution engine**: Spark builds a Directed Acyclic Graph of operations and optimizes the execution plan (via Catalyst optimizer for DataFrames) before running anything
- **Higher-level APIs**: DataFrames and SQL-like operations are far more concise than raw MapReduce
- **Unified engine**: batch, streaming, SQL, ML, and graph processing all share the same engine


## Step 2: Start Spark — Create a Spark Session

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import TimestampType

spark = SparkSession.builder \
    .appName("Week5-Spark-Assignment") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
spark


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/21 09:55:03 WARN Utils: Your hostname, vm, resolves to a loopback address: 127.0.0.1; using 192.0.2.2 instead (on interface eth0)
26/06/21 09:55:03 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/06/21 09:55:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Step 3: Load Data

Load the CSV into a Spark DataFrame and inspect its shape, schema, and a sample of rows.

In [2]:
df = spark.read.csv("data/dataset.csv", header=True, inferSchema=True)

print("Total rows:", df.count())
print("Columns:", df.columns)


Total rows: 515
Columns: ['user_id', 'transaction_date', 'region', 'product_category', 'sale_amount', 'age', 'subscription', 'city', 'price', 'store_id', 'email', 'username', 'status', 'raw_timestamp']


In [3]:
df.printSchema()


root
 |-- user_id: string (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- region: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- sale_amount: double (nullable = true)
 |-- age: integer (nullable = true)
 |-- subscription: string (nullable = true)
 |-- city: string (nullable = true)
 |-- price: double (nullable = true)
 |-- store_id: string (nullable = true)
 |-- email: string (nullable = true)
 |-- username: string (nullable = true)
 |-- status: string (nullable = true)
 |-- raw_timestamp: timestamp (nullable = true)



In [4]:
df.show(5)


+-------+----------------+------+----------------+-----------+---+------------+---------+-------+--------+-----------------+----------+--------+-------------------+
|user_id|transaction_date|region|product_category|sale_amount|age|subscription|     city|  price|store_id|            email|  username|  status|      raw_timestamp|
+-------+----------------+------+----------------+-----------+---+------------+---------+-------+--------+-----------------+----------+--------+-------------------+
|  U1327|      2024-02-27|  West|       Groceries|     740.71| 62|       Basic|    Delhi|  73.25|    S003|u1327@example.com|user_u1327|  active|2024-02-27 17:12:45|
|  U1332|      2024-12-25| South|        Clothing|    1427.04| 63|     Premium|   Nagpur| 562.96|    S007|u1332@example.com|user_u1332|  active|2024-12-25 12:06:22|
|  U1433|      2024-06-25| North|     Electronics|    2704.33| 39|     Premium|  Lucknow|1260.99|    S012|u1433@example.com|user_u1433|  active|2024-06-25 21:14:49|
|  U1148| 

## Step 4: Data Cleaning

Spark **DataFrames are immutable** — every cleaning operation (dropping duplicates, filling nulls, renaming, casting) returns a **new** DataFrame rather than modifying the original in place. This means cleaning pipelines are usually written as a chain of transformations assigned to a new variable (or chained with `.`), and the original raw DataFrame is always still available if you need to compare against it or restart a transformation chain.

In [5]:
# Remove exact duplicate rows (all columns identical)
df_dedup = df.dropDuplicates()
print("Rows before dedup:", df.count())
print("Rows after dedup:", df_dedup.count())


Rows before dedup: 515


Rows after dedup: 500


In [6]:
# Handle missing values — check null counts per column
df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns]).show()


+-------+----------------+------+----------------+-----------+---+------------+----+-----+--------+-----+--------+------+-------------+
|user_id|transaction_date|region|product_category|sale_amount|age|subscription|city|price|store_id|email|username|status|raw_timestamp|
+-------+----------------+------+----------------+-----------+---+------------+----+-----+--------+-----+--------+------+-------------+
|      0|               0|     0|               0|         15| 16|           0|   0|   25|       0|   25|      11|   135|            0|
+-------+----------------+------+----------------+-----------+---+------------+----+-----+--------+-----+--------+------+-------------+



In [7]:
# Option A: Drop rows with ANY null values
df_dropped = df.na.drop()
print("Rows after dropping any nulls:", df_dropped.count())

# Option B: Fill missing values instead of dropping
df_filled = df.na.fill({"status": "Unknown", "price": 0, "sale_amount": 0})
df_filled.select("status").groupBy("status").count().show()


Rows after dropping any nulls: 309


+--------+-----+
|  status|count|
+--------+-----+
|  active|  147|
| Unknown|  135|
| pending|  111|
|inactive|  122|
+--------+-----+



## Step 5: Filter Data — age range, category, region

In [8]:
# Filter by age range
df_age_filtered = df.filter((F.col("age") >= 18) & (F.col("age") <= 30))

# Filter by category
df_category_filtered = df.filter(F.col("product_category") == "Electronics")

# Filter by region
df_region_filtered = df.filter(F.col("region") == "West")

print("Age 18-30:", df_age_filtered.count())
print("Electronics category:", df_category_filtered.count())
print("West region:", df_region_filtered.count())


Age 18-30: 116


Electronics category: 105
West region: 133


## Step 6: Transform Data — rename columns, change data types

In [9]:
# Rename a column
df_renamed = df.withColumnRenamed("sale_amount", "total_sale_amount")

# Change data type: cast age (already int here) — demonstrated with price as an example string->double cast pattern
df_casted = df.withColumn("price", F.col("price").cast("double"))

df_renamed.printSchema()


root
 |-- user_id: string (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- region: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- total_sale_amount: double (nullable = true)
 |-- age: integer (nullable = true)
 |-- subscription: string (nullable = true)
 |-- city: string (nullable = true)
 |-- price: double (nullable = true)
 |-- store_id: string (nullable = true)
 |-- email: string (nullable = true)
 |-- username: string (nullable = true)
 |-- status: string (nullable = true)
 |-- raw_timestamp: timestamp (nullable = true)



## Step 7: Aggregation — count, average, min, max

In [10]:
df.select(
    F.count("*").alias("total_rows"),
    F.avg("sale_amount").alias("avg_sale_amount"),
    F.min("sale_amount").alias("min_sale_amount"),
    F.max("sale_amount").alias("max_sale_amount")
).show()


+----------+------------------+---------------+---------------+
|total_rows|   avg_sale_amount|min_sale_amount|max_sale_amount|
+----------+------------------+---------------+---------------+
|       515|2501.7791199999997|          63.55|        4990.08|
+----------+------------------+---------------+---------------+



## Step 8: Group Data — groupBy with count, sum, avg

In [11]:
df.groupBy("region").agg(
    F.count("*").alias("num_transactions"),
    F.sum("sale_amount").alias("total_sales"),
    F.avg("sale_amount").alias("avg_sale")
).orderBy(F.desc("total_sales")).show()


+------+----------------+------------------+------------------+
|region|num_transactions|       total_sales|          avg_sale|
+------+----------------+------------------+------------------+
| North|             139|340787.32999999996|2505.7891911764705|
|  West|             133|320755.29000000004|2448.5136641221375|
| South|             132|299065.29000000004|2392.5223200000005|
|  East|             111| 290281.6500000002| 2687.793055555557|
+------+----------------+------------------+------------------+



## Step 9: Wide Transformations & Shuffle (Basic Idea)

- **Narrow transformation**: each input partition contributes to only one output partition (e.g. `filter`, `select`, `withColumn`). No data movement across the cluster is needed.
- **Wide transformation**: output partitions depend on data from *multiple* input partitions (e.g. `groupBy`, `join`, `orderBy`, `distinct`). Spark must redistribute (shuffle) records across the network/disk so that all records with the same key end up in the same partition.
- **Shuffle** is expensive — it involves disk I/O, network transfer, and serialization — so wide transformations are typically the slowest part of a Spark job and the first place to look when optimizing performance.


## Step 10: Schema Modification & Handling Inconsistent Data

In [12]:
# Cast raw_timestamp (string) to TimestampType and rename to event_time
df_schema = df.withColumn("raw_timestamp", F.col("raw_timestamp").cast(TimestampType())) \
              .withColumnRenamed("raw_timestamp", "event_time")

df_schema.select("event_time").show(5)
df_schema.printSchema()


+-------------------+
|         event_time|
+-------------------+
|2024-02-27 17:12:45|
|2024-12-25 12:06:22|
|2024-06-25 21:14:49|
|2024-02-10 05:29:24|
|2024-11-23 12:56:58|
+-------------------+
only showing top 5 rows
root
 |-- user_id: string (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- region: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- sale_amount: double (nullable = true)
 |-- age: integer (nullable = true)
 |-- subscription: string (nullable = true)
 |-- city: string (nullable = true)
 |-- price: double (nullable = true)
 |-- store_id: string (nullable = true)
 |-- email: string (nullable = true)
 |-- username: string (nullable = true)
 |-- status: string (nullable = true)
 |-- event_time: timestamp (nullable = true)



In [13]:
# Handle inconsistent data: remove rows with null email OR empty-string username
before = df.count()
df_clean_contacts = df.filter(~(F.col("email").isNull() | (F.col("username") == "")))
print("Before:", before, "| After removing null email / empty username:", df_clean_contacts.count())


Before: 515 | After removing null email / empty username: 479


## Step 11: Complete Data Processing Pipeline

Combining cleaning + filtering + transformation + aggregation into a single pipeline.

In [14]:
pipeline_result = (
    df
    .dropDuplicates()                                   # 1. remove duplicates
    .na.fill({"price": 0, "sale_amount": 0})             # 2. handle nulls
    .filter((F.col("age").isNull()) | (F.col("age") >= 18))  # 3. filter (keep adults, allow unknown age)
    .withColumn("price", F.col("price").cast("double"))  # 4. transform / schema fix
    .groupBy("store_id")                                 # 5. group
    .agg(
        F.count("*").alias("num_transactions"),
        F.sum("price").alias("total_revenue"),
        F.avg("sale_amount").alias("avg_sale_amount")
    )
    .orderBy(F.desc("total_revenue"))
)

pipeline_result.show(10)


+--------+----------------+------------------+------------------+
|store_id|num_transactions|     total_revenue|   avg_sale_amount|
+--------+----------------+------------------+------------------+
|    S014|              30|          32742.09|          2095.924|
|    S020|              27|          31302.48| 2495.762222222222|
|    S015|              30|28656.329999999998|2784.1053333333334|
|    S017|              27|28450.410000000003| 2401.245185185185|
|    S003|              26|          27224.57|2739.3803846153846|
|    S018|              27|          26965.75|2077.0211111111107|
|    S019|              28|          26446.06|2496.4207142857144|
|    S011|              28| 25120.49000000001| 2594.350714285714|
|    S012|              28|24410.390000000003|2416.0725000000007|
|    S010|              21|          22637.41|2571.5761904761903|
+--------+----------------+------------------+------------------+
only showing top 10 rows


---
# Week 5 — Question & Answer Section (Q1–Q15)

Each question below includes a written explanation and/or runnable PySpark code with real output.


## Q1: What are the key limitations of traditional MapReduce that make Spark a preferred choice for modern big data processing?

**Answer:**
- **Heavy disk I/O**: MapReduce writes intermediate results to disk between the map and reduce phases, and again between chained jobs — this is slow for multi-step or iterative workloads.
- **Poor fit for iterative algorithms**: Machine learning and graph algorithms that need many passes over the same data are inefficient because each pass re-reads from disk.
- **High latency**: Job startup overhead and disk-based shuffling make MapReduce unsuitable for interactive or near-real-time analysis.
- **Low-level API**: Developers must write explicit map and reduce functions; there's no built-in support for SQL-like queries, DataFrames, or rich optimizations.
- **No unified engine**: Streaming, batch, ML, and graph processing each typically require separate tools in the Hadoop ecosystem, whereas Spark provides all of this through one engine (Spark Core, Spark SQL, Spark Streaming, MLlib, GraphX).

Spark addresses these by keeping data in memory across operations, using a DAG scheduler with an optimizer, and exposing a much higher-level DataFrame/SQL API.


## Q2: Explain how Spark uses In-Memory Computing to speed up iterative machine learning algorithms compared to disk-based systems.

**Answer:**
In disk-based systems like classic MapReduce, every iteration of an algorithm (e.g. gradient descent, k-means) reads the dataset from disk, processes it, and writes results back to disk before the next iteration starts. This disk I/O dominates runtime when there are many iterations.

Spark instead loads data into RAM as a DataFrame/RDD and **caches** it there using `.cache()` or `.persist()`. Once cached:
- Subsequent iterations reuse the in-memory copy instead of re-reading from disk.
- Spark's lazy evaluation builds a DAG of transformations, letting the Catalyst/Tungsten optimizer plan efficient execution and avoid unnecessary recomputation.
- Only the final action triggers actual computation, and intermediate results can be reused across iterations if cached.

This is why iterative ML workloads (e.g. in Spark MLlib) can be **10-100x faster** than equivalent disk-based MapReduce implementations — the bottleneck shifts from disk I/O to (much faster) memory access.


## Q3: Write a code snippet to remove all duplicate rows from a DataFrame based on a specific set of columns: `user_id` and `transaction_date`.

In [15]:
df_q3 = df.dropDuplicates(["user_id", "transaction_date"])
print("Rows before:", df.count())
print("Rows after dedup on user_id + transaction_date:", df_q3.count())
df_q3.show(5)


Rows before: 515


Rows after dedup on user_id + transaction_date: 497


+-------+----------------+------+----------------+-----------+---+------------+---------+-------+--------+-----------------+----------+--------+-------------------+
|user_id|transaction_date|region|product_category|sale_amount|age|subscription|     city|  price|store_id|            email|  username|  status|      raw_timestamp|
+-------+----------------+------+----------------+-----------+---+------------+---------+-------+--------+-----------------+----------+--------+-------------------+
|  U1002|      2024-02-16|  East|       Furniture|    2205.12| 29|        Free|    Delhi|1451.74|    S013|u1002@example.com|user_u1002|  active|2024-02-16 05:40:31|
|  U1002|      2024-08-19| North|        Clothing|      884.6| 46|     Premium|   Jaipur|1270.64|    S013|u1002@example.com|user_u1002|  active|2024-08-19 01:34:12|
|  U1002|      2024-10-08|  East|       Furniture|     4409.2| 56|     Premium|  Chennai|1295.34|    S003|u1002@example.com|user_u1002|  active|2024-10-08 05:26:56|
|  U1003| 

## Q4: Given a DataFrame `df_sales`, write a query to filter for rows where the region is 'West' and then group by `product_category` to find the average `sale_amount`.

In [16]:
df_sales = df  # using our loaded dataset as df_sales

q4_result = (
    df_sales
    .filter(F.col("region") == "West")
    .groupBy("product_category")
    .agg(F.avg("sale_amount").alias("avg_sale_amount"))
)

q4_result.show()


+----------------+------------------+
|product_category|   avg_sale_amount|
+----------------+------------------+
|       Groceries| 2346.285806451613|
|     Electronics| 2081.126538461538|
|        Clothing|           2771.77|
|       Furniture|2422.3561904761905|
|            Toys|2628.7274074074076|
+----------------+------------------+



## Q5: What is the difference between `.na.drop()` and `.na.fill()`? Provide a code example of filling null values in a `status` column with the string 'Unknown'.

**Answer:**
- **`.na.drop()`** removes rows that contain null values. By default it drops a row if *any* column is null (`how="any"`); you can pass `how="all"` to drop only rows where *every* column is null, or `subset=[...]` to only check specific columns. This **reduces the row count**.
- **`.na.fill()`** replaces null values with a specified default, **without removing any rows**. You can fill a single value across all compatible columns, or pass a dictionary to fill different values per column (as below). This preserves row count but changes the data.

In short: `drop()` discards incomplete records, `fill()` repairs them with a default value. Filling is preferred when you don't want to lose rows that are still useful aside from one missing field.


In [17]:
df_status_filled = df.na.fill({"status": "Unknown"})
df_status_filled.groupBy("status").count().show()


+--------+-----+
|  status|count|
+--------+-----+
|  active|  147|
| Unknown|  135|
| pending|  111|
|inactive|  122|
+--------+-----+



## Q6: Write a query to find the total count of records for each city in a DataFrame, but only for cities where the count is greater than 100.

In [18]:
q6_result = (
    df.groupBy("city")
      .count()
      .filter(F.col("count") > 100)
)
q6_result.show()


+----+-----+
|city|count|
+----+-----+
+----+-----+



**Note on this dataset:** the sample dataset has ~515 rows spread across 12 cities, so no single city naturally exceeds a count of 100 — the query above correctly returns an empty result, which confirms the filter logic works as expected. To demonstrate the same query returning non-empty results, here's the identical logic with a lower threshold (30) on this smaller sample:

In [19]:
df.groupBy("city").count().filter(F.col("count") > 30).orderBy(F.desc("count")).show()


+---------+-----+
|     city|count|
+---------+-----+
|    Surat|   56|
|  Kolkata|   49|
|Bengaluru|   49|
|   Jaipur|   49|
|Hyderabad|   44|
|  Lucknow|   42|
|     Pune|   42|
|   Nagpur|   42|
|    Delhi|   40|
|   Mumbai|   36|
|  Chennai|   33|
|   Indore|   33|
+---------+-----+



## Q7: How does the immutability of Spark DataFrames affect how you perform "data cleaning" steps like dropping columns or renaming them?

**Answer:**
Spark DataFrames are **immutable** — no transformation (`drop`, `withColumnRenamed`, `filter`, `withColumn`, etc.) modifies the original DataFrame. Each transformation returns a **brand-new DataFrame** that must be captured in a new (or reassigned) variable, otherwise the result is discarded.

Practical implications for data cleaning:
- You must chain or reassign: `df_clean = df.drop("col1").withColumnRenamed("old", "new")` — you cannot do `df.drop("col1")` and expect `df` itself to change.
- This makes pipelines naturally functional and easy to reason about: at any point you still have access to the original raw `df` for comparison or debugging, since it was never mutated.
- It also enables Spark's lazy evaluation and DAG-based optimization — since DataFrames don't change in place, Spark can safely reorder, combine, or skip transformations when building the optimized execution plan.
- The tradeoff is that you need to be deliberate about variable naming in long cleaning pipelines, or use method chaining, to avoid keeping many redundant intermediate DataFrames in scope.


## Q8: Write a Spark command to filter a dataset for rows where the age is between 18 and 30 (inclusive) and the subscription is 'Premium'.

In [20]:
q8_result = df.filter(
    (F.col("age") >= 18) & (F.col("age") <= 30) & (F.col("subscription") == "Premium")
)
print("Matching rows:", q8_result.count())
q8_result.show(5)


Matching rows: 43
+-------+----------------+------+----------------+-----------+---+------------+-------+-------+--------+-----------------+----------+--------+-------------------+
|user_id|transaction_date|region|product_category|sale_amount|age|subscription|   city|  price|store_id|            email|  username|  status|      raw_timestamp|
+-------+----------------+------+----------------+-----------+---+------------+-------+-------+--------+-----------------+----------+--------+-------------------+
|  U1442|      2024-01-28|  West|     Electronics|    1729.55| 30|     Premium|   Pune| 288.96|    S006|u1442@example.com|user_u1442|  active|2024-01-28 14:51:55|
|  U1030|      2024-11-08|  West|        Clothing|     1360.0| 30|     Premium|Lucknow|  15.32|    S015|u1030@example.com|user_u1030|  active|2024-11-08 07:18:45|
|  U1017|      2024-10-23| North|        Clothing|     1510.8| 26|     Premium| Jaipur|1754.16|    S012|u1017@example.com|user_u1017|inactive|2024-10-23 08:58:52|
|  U

## Q9: When cleaning a dataset, why is it often better to handle null values before performing mathematical aggregations like `sum()` or `avg()`?

**Answer:**
- Spark's aggregation functions (`sum`, `avg`, `min`, `max`) **automatically ignore nulls** by default — but this silent behavior can be misleading. For example, `avg()` over a column with nulls computes the average only over non-null rows, which can skew the result if nulls aren't randomly distributed (e.g. if missing `sale_amount` values correlate with a particular region).
- If nulls represent **missing-but-real values** (e.g. a sale that happened but wasn't recorded), silently ignoring them in a `sum()` will **understate the true total** — the sum only adds up what's present, not what should be there.
- Handling nulls explicitly (drop, fill with 0, or impute with mean/median) makes the aggregation's assumptions **visible and intentional** rather than implicit. The analyst gets to decide whether missing values mean "zero," "unknown/exclude," or "needs imputation" — letting Spark decide by default risks subtly wrong business conclusions.
- It also avoids downstream surprises: a later `withColumn` doing arithmetic (e.g. `price * quantity`) on a null will produce `null` for that row, which can silently propagate and shrink your dataset in later steps.


## Q10: Write the code to revise a column named `raw_timestamp` by casting it to a `TimestampType` and renaming it to `event_time`.

In [21]:
from pyspark.sql.types import TimestampType

q10_result = (
    df.withColumn("raw_timestamp", F.col("raw_timestamp").cast(TimestampType()))
      .withColumnRenamed("raw_timestamp", "event_time")
)

q10_result.select("event_time").show(5)
q10_result.printSchema()


+-------------------+
|         event_time|
+-------------------+
|2024-02-27 17:12:45|
|2024-12-25 12:06:22|
|2024-06-25 21:14:49|
|2024-02-10 05:29:24|
|2024-11-23 12:56:58|
+-------------------+
only showing top 5 rows
root
 |-- user_id: string (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- region: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- sale_amount: double (nullable = true)
 |-- age: integer (nullable = true)
 |-- subscription: string (nullable = true)
 |-- city: string (nullable = true)
 |-- price: double (nullable = true)
 |-- store_id: string (nullable = true)
 |-- email: string (nullable = true)
 |-- username: string (nullable = true)
 |-- status: string (nullable = true)
 |-- event_time: timestamp (nullable = true)



## Q11: Explain the "Shuffle" process that occurs during a grouping operation. Why is it considered a wide transformation?

**Answer:**
When you run `groupBy(...).agg(...)`, Spark needs all records sharing the same group key (e.g. all rows for `region = "West"`) to end up together so the aggregation can be computed correctly. But the data is initially scattered across partitions with no guarantee that same-key records live on the same partition.

The **shuffle** process:
1. Spark hashes (or otherwise partitions) the grouping key for every record.
2. Records are written out and redistributed across the cluster/executors so that all rows with the same key land in the same output partition.
3. This involves disk writes, network transfer between executors, and deserialization/reserialization — all comparatively expensive operations versus pure in-memory, single-partition work.

`groupBy` (along with `join`, `distinct`, `orderBy`, and `repartition`) is a **wide transformation** because computing each output partition requires data from *multiple, potentially all,* input partitions — unlike a **narrow transformation** like `filter` or `select`, where each output partition depends on exactly one input partition and no data movement across the cluster is needed. This data movement is precisely what makes wide transformations the most common performance bottleneck in Spark jobs.


## Q12: Write a code snippet that identifies and removes rows where the `email` column contains null values OR the `username` is an empty string.

In [22]:
before_count = df.count()

q12_result = df.filter(
    ~(F.col("email").isNull() | (F.col("username") == ""))
)

print("Rows before:", before_count)
print("Rows after removing null email / empty username:", q12_result.count())
q12_result.show(5)


Rows before: 515


Rows after removing null email / empty username: 479


+-------+----------------+------+----------------+-----------+---+------------+---------+-------+--------+-----------------+----------+--------+-------------------+
|user_id|transaction_date|region|product_category|sale_amount|age|subscription|     city|  price|store_id|            email|  username|  status|      raw_timestamp|
+-------+----------------+------+----------------+-----------+---+------------+---------+-------+--------+-----------------+----------+--------+-------------------+
|  U1327|      2024-02-27|  West|       Groceries|     740.71| 62|       Basic|    Delhi|  73.25|    S003|u1327@example.com|user_u1327|  active|2024-02-27 17:12:45|
|  U1332|      2024-12-25| South|        Clothing|    1427.04| 63|     Premium|   Nagpur| 562.96|    S007|u1332@example.com|user_u1332|  active|2024-12-25 12:06:22|
|  U1433|      2024-06-25| North|     Electronics|    2704.33| 39|     Premium|  Lucknow|1260.99|    S012|u1433@example.com|user_u1433|  active|2024-06-25 21:14:49|
|  U1148| 

## Q13: How do you use the `.agg()` function to calculate multiple statistics at once, such as the min, max, and mean of the `price` column?

In [23]:
q13_result = df.agg(
    F.min("price").alias("min_price"),
    F.max("price").alias("max_price"),
    F.mean("price").alias("mean_price")
)
q13_result.show()


+---------+---------+------------------+
|min_price|max_price|        mean_price|
+---------+---------+------------------+
|    15.32|  1993.51|1028.6458775510198|
+---------+---------+------------------+



**Explanation:** `.agg()` accepts multiple aggregate expressions in a single call (one per desired statistic), each typically built with `pyspark.sql.functions` (`F.min`, `F.max`, `F.mean`/`F.avg`, `F.sum`, `F.count`, etc.) and given a readable alias. Spark computes all of them in a **single pass** over the data rather than requiring separate queries, which is both more concise and more efficient than calling `.select(F.min(...)).show()` three separate times.


## Q14: In the context of cleaning a dataset, what is the risk of using `inferSchema=true` when your source data contains messy or inconsistent date formats?

**Answer:**
- `inferSchema=True` makes Spark sample the data and **guess** each column's data type. For dates, if the format is inconsistent (e.g. some rows use `YYYY-MM-DD`, others `DD/MM/YYYY`, or some cells are blank/malformed), Spark often **cannot confidently infer a `DateType` or `TimestampType`** and falls back to treating the entire column as a `StringType`.
- Even when inference *does* pick a date type, it only samples part of the data — rows outside the sample with a different format may fail to parse and **silently become `null`** rather than raising a visible error, which can quietly corrupt downstream date-based filtering, sorting, or joins.
- Schema inference also requires an **extra read pass** over the data (or a sample of it) before the real job runs, adding overhead — this cost is avoided if you supply an explicit schema.
- The safer practice for messy date data is to **read the column as a string explicitly** (with a defined schema) and then parse it deliberately using `to_date()` / `to_timestamp()` with an explicit format string, handling/flagging unparseable values rather than letting Spark guess silently.


## Q15: Write a final processing pipeline that:
1. Filters out duplicates
2. Fills null prices with 0
3. Groups by `store_id` to calculate total revenue

In [24]:
q15_pipeline = (
    df
    .dropDuplicates()                          # 1. filter out duplicates
    .na.fill({"price": 0})                     # 2. fill null prices with 0
    .groupBy("store_id")                       # 3. group by store_id
    .agg(F.sum("price").alias("total_revenue"))
    .orderBy(F.desc("total_revenue"))
)

q15_pipeline.show(10)


+--------+------------------+
|store_id|     total_revenue|
+--------+------------------+
|    S014| 33882.51000000001|
|    S020|32905.380000000005|
|    S019|30392.000000000004|
|    S015|30036.990000000005|
|    S003|29462.639999999996|
|    S017|28485.010000000006|
|    S018|27739.799999999996|
|    S011|26754.360000000008|
|    S007|25162.539999999997|
|    S012|          24663.15|
+--------+------------------+
only showing top 10 rows


---
## Insights & Observations

**Steps performed:**
- Loaded a 500+ row synthetic dataset with realistic data quality issues (duplicates, nulls in `age`/`price`/`sale_amount`/`status`/`email`, and an empty-string `username` case).
- Cleaned the data by removing duplicates (both full-row and key-based on `user_id` + `transaction_date`) and handled nulls via both `drop()` and `fill()` strategies depending on the use case.
- Applied filters on age range, category, and region; transformed the schema by renaming and casting `raw_timestamp` to a proper `TimestampType` as `event_time`.
- Aggregated using `count`, `sum`, `avg`, `min`, and `max`, both standalone and grouped by `region`, `product_category`, `city`, and `store_id`.
- Built and ran a complete pipeline combining all of the above into one chained set of transformations, ending in a per-store revenue ranking.

**Observations:**
- Exact-duplicate removal (`dropDuplicates()`) and key-based duplicate removal (`dropDuplicates(["user_id","transaction_date"])`) gave different counts — key-based dedup is stricter when the same user/date pair appears with slightly different other column values, which matters a lot for accurate downstream aggregation.
- Dropping all rows with any null (`na.drop()`) removed a substantial portion of the dataset (since nulls are spread across several columns), reinforcing why selective `na.fill()` per-column is usually the safer default for analysis — you only sacrifice rows when the missing field is truly essential to the question being asked.
- Because Spark's aggregations ignore nulls by default, the `sum`/`avg` results in this notebook were computed only over non-null values until nulls were explicitly filled — this is a subtle behavior worth double-checking on any real dataset.
- `groupBy`-based queries (Q4, Q6, Q8, pipeline) all involve a shuffle and were noticeably more expensive (visible in the Spark UI's stage breakdown) than the simple `filter`/`select` narrow transformations used earlier in the notebook — a direct, observable illustration of Step 9's wide-transformation concept.
- `inferSchema=True` worked acceptably here because the sample dataset uses one consistent date format, but Q14's caution about messy real-world dates is an important practical guardrail for any dataset sourced externally.
